In [1]:
from fredapi import Fred
import pandas as pd
import numpy as np
import hvplot.pandas

f = Fred(api_key = 'd099e8a4f1b849feb319c554d18cb172')

# Retrieve the raw data from FRED
df = f.get_series('UNRATE', observation_start='2006-1-1', frequency = 'm')

# Plot
df.hvplot(title = "Unemployment Rate")

:Curve   [index]   (0)

In [2]:
df2 = f.get_series_first_release('UNRATE')

# use concat to combine the 2 Series types
combine = pd.concat([df,df2], join = 'inner', axis = 1)
combine.columns = ['unrevised', 'revised']

# Plot
combine.hvplot(title = 'Revised Vs Unrevised: Unemployment Rate', 
               y = ['unrevised', 'revised'], 
               color = ['blue','gray'], grid = True)

c:\Users\agung\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\indexes\base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [3]:
f.search("Australia CPI")

,id,realtime_start,realtime_end,title,observation_start,observation_end,frequency,frequency_short,units,units_short,seasonal_adjustment,seasonal_adjustment_short,last_updated,popularity,notes
series id,,,,,,,,,,,,,,,
AUSCPIALLQINMEI,AUSCPIALLQINMEI,2024-07-28,2024-07-28,"Consumer Price Indices (CPIs, HICPs), COICOP 1...",1948-07-01,2024-01-01,Quarterly,Q,Index 2015=100,Index 2015=100,Not Seasonally Adjusted,NSA,2024-05-15 13:59:09-05:00,22,OECD Data Filters: REF_AREA: AUS MEASURE: CPI...
AUSCPALTT01IXNBQ,AUSCPALTT01IXNBQ,2024-07-28,2024-07-28,Consumer Price Index: All Items: Total for Aus...,1960-01-01,2023-07-01,Quarterly,Q,Index source base,Index source base,Not Seasonally Adjusted,NSA,2023-11-17 12:00:08-06:00,4,OECD Descriptor ID: CPALTT01 OECD unit ID: IDX...
AUSCPIALLAINMEI,AUSCPIALLAINMEI,2024-07-28,2024-07-28,"Consumer Price Indices (CPIs, HICPs), COICOP 1...",1949-01-01,2023-01-01,Annual,A,Index 2015=100,Index 2015=100,Not Seasonally Adjusted,NSA,2024-05-15 14:02:29-05:00,3,OECD Data Filters: REF_AREA: AUS MEASURE: CPI...
CCRETT01AUM661N,CCRETT01AUM661N,2024-07-28,2024-07-28,Financial Market: Real Effective Exchange Rate...,1972-01-01,2024-06-01,Monthly,M,Index 2015=100,Index 2015=100,Not Seasonally Adjusted,NSA,2024-07-15 14:02:07-05:00,5,OECD Data Filters: REF_AREA: AUS MEASURE: CCR...
CCRETT01AUQ661N,CCRETT01AUQ661N,2024-07-28,2024-07-28,Financial Market: Real Effective Exchange Rate...,1972-01-01,2024-04-01,Quarterly,Q,Index 2015=100,Index 2015=100,Not Seasonally Adjusted,NSA,2024-07-15 14:09:50-05:00,1,OECD Data Filters: REF_AREA: AUS MEASURE: CCR...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
DDDI08AUA156NWDB,DDDI08AUA156NWDB,2024-07-28,2024-07-28,Financial System Deposits to GDP for Australia,1960-01-01,2021-01-01,Annual,A,Percent,%,Not Seasonally Adjusted,NSA,2024-05-07 15:33:35-05:00,1,"Demand, time and saving deposits in deposit mo..."
DDDI12AUA156NWDB,DDDI12AUA156NWDB,2024-07-28,2024-07-28,Private Credit by Deposit Money Banks and Othe...,1960-01-01,2021-01-01,Annual,A,Percent,%,Not Seasonally Adjusted,NSA,2024-05-07 15:33:22-05:00,1,Private credit by deposit money banks and othe...
DDDM07AUA156NWDB,DDDM07AUA156NWDB,2024-07-28,2024-07-28,Outstanding Total International Debt Securitie...,1980-01-01,2020-01-01,Annual,A,Percent,%,Not Seasonally Adjusted,NSA,2024-05-07 15:32:35-05:00,1,Amount of international debt securities (amoun...


In [4]:
columns = ['AUS','JAP','USA','GBR','KOR']

aus = f.get_series('NGDPRSAXDCAUQ')
jap = f.get_series('JPNRGDPEXP')
us = f.get_series('GDPC1')
uk = f.get_series('NGDPRSAXDCGBQ')
korea = f.get_series('NGDPRSAXDCKRQ')

#Combine all the single series, only take common dates using "inner"
gdp_global = pd.concat([aus, jap, us, uk, korea], join='inner', axis=1)

# Creare YoY Real GDP (divided by 4 because we have quarterly data) & Drop NA's
gdp_global = ((gdp_global / gdp_global.shift(4)) - 1).dropna(axis = 0)

gdp_global.columns = columns

# Plot
gdp_global.hvplot(title = 'Global GDP', grid = True)

:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [5]:
# Extract the 10Y Government Bond Yields for each country (make quarterly):
aus_y = f.get_series('IRLTLT01AUM156N', frequency = 'q')
jap_y = f.get_series('IRLTLT01JPM156N', frequency = 'q')
us_y = f.get_series('IRLTLT01USM156N', frequency = 'q')
uk_y = f.get_series('IRLTLT01GBM156N', frequency = 'q')
korea_y = f.get_series('IRLTLT01KRM156N', frequency = 'q')

# Combine again
global_10y_yields = pd.concat([aus_y, jap_y, us_y, uk_y, korea_y], join='inner', axis=1)

global_10y_yields.columns = columns

# Plot
global_10y_yields.hvplot(title = 'Global 10Y Yields', grid = True)

:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [6]:
# keys allows for multi-level columns
country_df = pd.concat([gdp_global, global_10y_yields], axis=1, 
                        keys=['GDP', '10Y Yields'])

country_df

GDP                                         10Y Yields  \
                 AUS       JAP       USA       GBR       KOR        AUS   
1995-01-01  0.030572  0.013003  0.034814  0.029104  0.096482        NaN   
1995-04-01  0.022767  0.028227  0.024024  0.021861  0.100693        NaN   
1995-07-01  0.037684  0.028477  0.026733  0.022922  0.104954        NaN   
1995-10-01  0.027342  0.035070  0.021998  0.021650  0.083054        NaN   
1996-01-01  0.043736  0.032171  0.026013  0.023116  0.081119        NaN   
...              ...       ...       ...       ...       ...        ...   
2023-04-01  0.019072  0.022171  0.023825  0.002090  0.008699   3.586333   
2023-07-01  0.020519  0.014684  0.029269  0.001678  0.012547   4.125000   
2023-10-01  0.015570  0.012472  0.031345 -0.002479  0.021970   4.463667   
2024-01-01  0.011112 -0.002808  0.029234  0.002054  0.031646   4.111333   
2024-04-01       NaN       NaN       NaN       NaN       NaN   4.274333   

                                                
                 JAP   USA       GBR       KOR  
1995-01-01       NaN   NaN       NaN       NaN  
1995-04-01       NaN   NaN       NaN       NaN  
1995-07-01       NaN   NaN       NaN       NaN  
1995-10-01       NaN   NaN       NaN       NaN  
1996-01-01       NaN   NaN       NaN       NaN  
...              ...   ...       ...       ...  
2023-04-01  0.403333  3.59  3.992767  3.444000  
2023-07-01  0.666667  4.15  4.462300  3.830000  
2023-10-01  0.743333  4.44  4.234600  3.859000  
2024-01-01  0.721667  4.16  4.029300  3.390667  
2024-04-01  0.996667  4.44  4.201500  3.477000  

[118 rows x 10 columns]

In [7]:
# Selects only the Korean variables.
korea = country_df.xs('KOR', level=1, axis=1)

# Show
korea

,GDP,10Y Yields
1995-01-01,0.096482,NaN
1995-04-01,0.100693,NaN
1995-07-01,0.104954,NaN
1995-10-01,0.083054,NaN
1996-01-01,0.081119,NaN
...,...,...
2023-04-01,0.008699,3.444000
2023-07-01,0.012547,3.830000
2023-10-01,0.021970,3.859000
2024-01-01,0.031646,3.390667
